# IMDB Dataset Exploration

Load the Kaggle *Bag of Words Meets Bags of Popcorn* data using the project's shared configuration and helpers, then establish a basic understanding of its quality and structure before modeling.

## Imports

In [ ]:
# standard library
from pathlib import Path
import sys

# third-party
import pandas as pd
from IPython.display import display

# local
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from imdb_dataset_helpers import (
    build_cleanup_report,
    build_dataset_overview,
    clean_reviews,
    duplicate_review_examples,
    load_datasets,
    print_summary,
)

## Acquire and load the data

The shared helper downloads the Kaggle competition bundle only when the configured files are absent.

In [ ]:
train, test = load_datasets()

In [ ]:
print_summary("Labeled training data", train)
print_summary("Test data", test)

## Structure and data quality

### Display basic data: shapes, missing values, duplicates

In [ ]:
build_dataset_overview({"train": train, "test": test})

### Display some examples of duplicate entries
- 96 duplicates = 96 copies of the exact same movie review text string.
- 96 copies above and beyond the original movie review text string(s) from which they are duplicated
- we don't know if it's:
- 1 original copy and 96 clones OR
- 96 originals which each have been duplicated once
- we just know there are 96 copies/duplicates

for our strategy, we will take the original copy and remove the dupes.

In [ ]:
print("Two exact duplicate-review groups from the training data:")
display(duplicate_review_examples(train))

print("Two exact duplicate-review groups from the test data:")
display(duplicate_review_examples(test))

## Positive and Negative Sentiment: Counts and Percentage

In [ ]:
label_counts = train["sentiment"].value_counts().sort_index().rename("count").to_frame()
label_counts["percent"] = (100 * label_counts["count"] / len(train)).round(2)
label_counts.index.name = "sentiment"
label_counts

## Review lengths

Character and word counts give an early indication of sequence-length requirements and unusually short or long reviews.

In [ ]:
review_lengths = pd.DataFrame(
    {
        "train_characters": train["review"].str.len(),
        "train_words": train["review"].str.split().str.len(),
        "test_characters": test["review"].str.len(),
        "test_words": test["review"].str.split().str.len(),
    }
)
review_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

In [ ]:
longest_reviews = (
    train.assign(word_count=review_lengths["train_words"])
    .nlargest(5, "word_count")
    .loc[:, ["id", "sentiment", "word_count", "review"]]
)
display(longest_reviews)

## Initial cleanup

Create cleaned copies for later modeling while preserving `train` and `test` as the untouched raw data. The shared `clean_reviews()` helper parses and removes HTML, normalizes whitespace, keeps the first occurrence of each review, and removes only later duplicates. Duplicate detection happens after HTML cleanup, so reviews that become identical after normalization are also deduplicated.

In [ ]:
train_clean = clean_reviews(train)
test_clean = clean_reviews(test)

cleanup_report = build_cleanup_report(
    raw_datasets={"train": train, "test": test},
    cleaned_datasets={"train": train_clean, "test": test_clean},
)
cleanup_report

## Next steps

Potential follow-up work:

1. Establish a stratified validation split from `train_clean`.
2. Build a TF-IDF plus logistic-regression baseline.
3. Record validation metrics and error examples.
4. Compare the baseline with embedding-based or transformer models.